# DATA266 Task 2: Yelp Polarity Sentiment Classification

This notebook is my individual Task 2 work.

Requirements covered:

- Analyze review lengths, class distribution, and class balance.
- Preprocess and tokenize reviews.
- Learn embeddings from scratch.
- Train three different models: baseline and two experiments.
- Use no pretrained embeddings and no pretrained language models.
- Report the required classification, calibration, robustness, and hardware metrics.
- Review 20 model errors and propose testable fixes.

In [1]:
import sys
import subprocess
import importlib.util

required_packages = {
    "numpy": "numpy",
    "torch": "torch",
    "datasets": "datasets",
    "sklearn": "scikit-learn",
}

for import_name, package_name in required_packages.items():
    package_available = (
        importlib.util.find_spec(import_name) is not None
    )

    if package_available:
        print(f"{package_name} is already installed.")
    else:
        print(f"Installing {package_name}...")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            package_name
        ])

print("\nAll required packages are ready.")

numpy is already installed.
torch is already installed.
datasets is already installed.
scikit-learn is already installed.

All required packages are ready.


In [2]:
import os
import sys
import re
import csv
import json
import math
import time
import random
import platform
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    brier_score_loss,
    f1_score,
)

SID4 = 4349
SEED = SID4
SLICE = SID4 % 1000
HP_ID = 5
CLS_A = 9
CLS_B = 2
TRAIN_SEEDS = [4349, 4350, 4351]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONFIG = {
    "max_train_examples": None,
    "max_test_examples": None,
    "validation_fraction": 0.10,
    "vocab_size": 30_000,
    "min_frequency": 2,
    "max_length": 256,
    "batch_size": 256,
    "epochs": 5,
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "gradient_clip": 1.0,
    "bootstrap_samples": 1000,
    "seed": SEED,
}

print("DATA266 Task 2 - Yelp Sentiment Classification")
print(f"SID4: {SID4}")
print(f"SEED: {SEED}")
print(f"SLICE: {SLICE}")
print(f"HP_ID: {HP_ID}")
print(f"CLS_A: {CLS_A}")
print(f"CLS_B: {CLS_B}")
print(f"Training seeds: {TRAIN_SEEDS}")
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

C:\Users\yashashree2\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DATA266 Task 2 - Yelp Sentiment Classification
SID4: 4349
SEED: 4349
SLICE: 349
HP_ID: 5
CLS_A: 9
CLS_B: 2
Training seeds: [4349, 4350, 4351]
Python: 3.13.3
PyTorch: 2.14.0+cu132
NumPy: 2.5.3
Device: cuda
GPU: NVIDIA GeForce RTX 4090


## Load the Yelp Polarity Dataset

The dataset contains binary sentiment labels:

- `0`: negative review
- `1`: positive review

The official training split will be divided into training and validation data. The official test split remains untouched for final evaluation.

In [3]:
yelp = load_dataset("fancyzhx/yelp_polarity")

train_table = yelp["train"]
test_table = yelp["test"]

train_texts = list(train_table["text"])
train_labels = [int(value) for value in train_table["label"]]
test_texts = list(test_table["text"])
test_labels = [int(value) for value in test_table["label"]]

if CONFIG["max_train_examples"] is not None:
    train_texts = train_texts[:CONFIG["max_train_examples"]]
    train_labels = train_labels[:CONFIG["max_train_examples"]]

if CONFIG["max_test_examples"] is not None:
    test_texts = test_texts[:CONFIG["max_test_examples"]]
    test_labels = test_labels[:CONFIG["max_test_examples"]]

print("Training reviews:", len(train_texts))
print("Test reviews:", len(test_texts))
print("Columns:", train_table.column_names)
print("Example label:", train_labels[0])
print("Example review:", train_texts[0][:500])

Training reviews: 560000
Test reviews: 38000
Columns: ['text', 'label']
Example label: 0
Example review: Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff.  It seems that his staff simply never answers the phone.  It usually takes 2 hours of repeated calling to get an answer.  Who has time for that or wants to deal with it?  I have run into this problem with many other doctors and I just don't get it.  You have office workers, you have patients with medical needs, why isn't anyone an


In [4]:
# Explicit missing and malformed data check

def check_data_quality(texts, labels, split_name):
    missing_text = sum(text is None for text in texts)
    non_string_text = sum(
        not isinstance(text, str)
        for text in texts
        if text is not None
    )
    empty_text = sum(
        isinstance(text, str) and len(text.strip()) == 0
        for text in texts
    )
    invalid_labels = sum(label not in [0, 1] for label in labels)

    print(f"{split_name} data quality")
    print("-" * 40)
    print("Total examples:", len(texts))
    print("Missing text values:", missing_text)
    print("Non-string text values:", non_string_text)
    print("Empty text values:", empty_text)
    print("Invalid labels:", invalid_labels)

    return {
        "missing_text": missing_text,
        "non_string_text": non_string_text,
        "empty_text": empty_text,
        "invalid_labels": invalid_labels,
    }


train_quality = check_data_quality(train_texts, train_labels, "Training")
test_quality = check_data_quality(test_texts, test_labels, "Test")


Training data quality
----------------------------------------
Total examples: 560000
Missing text values: 0
Non-string text values: 0
Empty text values: 0
Invalid labels: 0
Test data quality
----------------------------------------
Total examples: 38000
Missing text values: 0
Non-string text values: 0
Empty text values: 0
Invalid labels: 0


## Text Preprocessing and Dataset Analysis

Reviews are lowercased, punctuation is normalized, and word tokens are created. Negation words such as `not`, `no`, and `never` are preserved because removing them would damage sentiment information.

In [5]:
NEGATION_WORDS = {"not", "no", "never", "neither", "nor", "nothing", "hardly"}

COMMON_STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "if", "then", "than", "of",
    "to", "in", "on", "for", "with", "at", "by", "from", "as", "is",
    "am", "are", "was", "were", "be", "been", "being", "it", "this",
    "that", "these", "those", "i", "me", "my", "we", "our", "you", "your",
    "he", "she", "they", "their", "them", "his", "her", "its", "do", "does",
    "did", "have", "has", "had", "will", "would", "can", "could", "should",
    "very", "there", "here", "so", "too", "also"
} - NEGATION_WORDS


def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9!?']+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    tokens = [
        token for token in tokens
        if token not in COMMON_STOPWORDS or token in NEGATION_WORDS
    ]
    return tokens


def review_length_statistics(texts, labels):
    lengths = [len(text.split()) for text in texts]
    class_counts = Counter(labels)

    print("Review count:", len(texts))
    print("Minimum word length:", min(lengths))
    print("Maximum word length:", max(lengths))
    print("Average word length:", sum(lengths) / len(lengths))
    print("Median word length:", sorted(lengths)[len(lengths) // 2])
    print("Class counts:", dict(class_counts))
    print("Class proportions:")
    for label, count in sorted(class_counts.items()):
        print(label, count / len(labels))

    return lengths


train_raw_lengths = review_length_statistics(
    train_texts,
    train_labels
)

test_raw_lengths = review_length_statistics(
    test_texts,
    test_labels
)

processed_train_tokens = [
    preprocess_text(text)
    for text in train_texts
]

processed_test_tokens = [
    preprocess_text(text)
    for text in test_texts
]

print("\nProcessed first review tokens:")
print(processed_train_tokens[0][:40])

Review count: 560000
Minimum word length: 1
Maximum word length: 1052
Average word length: 133.0288732142857
Median word length: 97
Class counts: {0: 280000, 1: 280000}
Class proportions:
0 0.5
1 0.5
Review count: 38000
Minimum word length: 1
Maximum word length: 1007
Average word length: 132.55863157894737
Median word length: 97
Class counts: {1: 19000, 0: 19000}
Class proportions:
0 0.5
1 0.5

Processed first review tokens:
['unfortunately', 'frustration', 'dr', "goldberg's", 'patient', 'repeat', 'experience', "i've", 'many', 'other', 'doctors', 'nyc', 'good', 'doctor', 'terrible', 'staff', 'seems', 'staff', 'simply', 'never', 'answers', 'phone', 'usually', 'takes', '2', 'hours', 'repeated', 'calling', 'get', 'answer', 'who', 'time', 'wants', 'deal', 'it?', 'run', 'into', 'problem', 'many', 'other']


## Create an Individual Train/Validation Split and Vocabulary

The validation split is stratified manually so both sentiment classes remain balanced. The vocabulary is learned only from the training portion. All embeddings are randomly initialized and learned during training.

In [6]:
def stratified_split(tokens, labels, validation_fraction, seed):
    indices_by_label = defaultdict(list)

    for index, label in enumerate(labels):
        indices_by_label[label].append(index)

    rng = random.Random(seed)
    train_indices = []
    validation_indices = []

    for label, label_indices in indices_by_label.items():
        label_indices = list(label_indices)
        rng.shuffle(label_indices)
        validation_count = int(
            len(label_indices) * validation_fraction
        )
        validation_indices.extend(
            label_indices[:validation_count]
        )
        train_indices.extend(
            label_indices[validation_count:]
        )

    rng.shuffle(train_indices)
    rng.shuffle(validation_indices)

    return train_indices, validation_indices


train_indices, validation_indices = stratified_split(
    processed_train_tokens,
    train_labels,
    CONFIG["validation_fraction"],
    SEED
)

train_tokens = [processed_train_tokens[index] for index in train_indices]
validation_tokens = [processed_train_tokens[index] for index in validation_indices]
train_split_labels = [train_labels[index] for index in train_indices]
validation_split_labels = [train_labels[index] for index in validation_indices]

token_counter = Counter()
for tokens in train_tokens:
    token_counter.update(tokens)

special_tokens = ["<PAD>", "<UNK>"]
vocabulary_words = [
    word for word, frequency in token_counter.most_common()
    if frequency >= CONFIG["min_frequency"]
]
vocabulary_words = vocabulary_words[:CONFIG["vocab_size"] - len(special_tokens)]

vocabulary = special_tokens + vocabulary_words
word_to_idx = {
    word: index for index, word in enumerate(vocabulary)
}
idx_to_word = {
    index: word for word, index in word_to_idx.items()
}

PAD_ID = word_to_idx["<PAD>"]
UNK_ID = word_to_idx["<UNK>"]
VOCAB_SIZE = len(vocabulary)

print("Training reviews:", len(train_tokens))
print("Validation reviews:", len(validation_tokens))
print("Vocabulary size:", VOCAB_SIZE)
print("Most common words:", token_counter.most_common(20))

Training reviews: 504000
Validation reviews: 56000
Vocabulary size: 30000
Most common words: [('n', 735150), ('not', 454535), ('food', 290449), ('place', 280450), ('good', 263186), ('out', 239797), ('like', 236033), ('all', 234927), ('just', 232758), ('get', 212896), ('one', 211186), ('when', 193834), ('up', 191366), ('time', 190501), ('great', 182650), ('service', 182104), ('about', 174736), ('no', 169959), ('back', 168383), ('go', 168294)]


In [7]:
def encode_tokens(tokens, word_to_idx, max_length):
    token_ids = [
        word_to_idx.get(token, UNK_ID)
        for token in tokens[:max_length]
    ]

    true_length = len(token_ids)

    if len(token_ids) < max_length:
        token_ids.extend(
            [PAD_ID] * (max_length - len(token_ids))
        )

    return token_ids, true_length


class YelpReviewDataset(Dataset):
    def __init__(self, token_lists, labels):
        self.token_lists = token_lists
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        input_ids, true_length = encode_tokens(
            self.token_lists[index],
            word_to_idx,
            CONFIG["max_length"]
        )

        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(self.labels[index], dtype=torch.long),
            torch.tensor(true_length, dtype=torch.long),
            self.token_lists[index]
        )


def review_collate(batch):
    input_ids, labels, lengths, token_lists = zip(*batch)

    return (
        torch.stack(input_ids),
        torch.stack(labels),
        torch.stack(lengths),
        list(token_lists)
    )


train_dataset = YelpReviewDataset(
    train_tokens,
    train_split_labels
)
validation_dataset = YelpReviewDataset(
    validation_tokens,
    validation_split_labels
)
test_dataset = YelpReviewDataset(
    processed_test_tokens,
    test_labels
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=review_collate
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=review_collate
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=review_collate
)

sample_batch = next(iter(train_loader))
print("Input batch shape:", tuple(sample_batch[0].shape))
print("Label batch shape:", tuple(sample_batch[1].shape))
print("Number of training batches:", len(train_loader))

Input batch shape: (256, 256)
Label batch shape: (256,)
Number of training batches: 1969


## Three From-Scratch Sentiment Models

The three models intentionally use different architectures:

1. **Baseline:** learned embeddings with masked mean pooling.
2. **Experiment A:** learned embeddings with multiple 1-D convolution widths.
3. **Experiment B:** learned embeddings with a bidirectional GRU.

None of the models use pretrained embeddings or pretrained language models.

In [8]:
class MeanPoolingClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_ID
        )
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, input_ids, lengths):
        embeddings = self.embedding(input_ids)
        mask = (input_ids != PAD_ID).unsqueeze(-1)
        masked_embeddings = embeddings * mask
        denominator = mask.sum(dim=1).clamp(min=1)
        pooled = masked_embeddings.sum(dim=1) / denominator
        return self.classifier(pooled)


class CNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=160, channels=128):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_ID
        )
        self.convolutions = nn.ModuleList([
            nn.Conv1d(embedding_dim, channels, kernel_size=3),
            nn.Conv1d(embedding_dim, channels, kernel_size=5),
            nn.Conv1d(embedding_dim, channels, kernel_size=7)
        ])
        self.classifier = nn.Sequential(
            nn.Linear(channels * 3, 192),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(192, 2)
        )

    def forward(self, input_ids, lengths):
        embeddings = self.embedding(input_ids).transpose(1, 2)
        pooled_features = []

        for convolution in self.convolutions:
            features = F.relu(convolution(embeddings))
            pooled = F.adaptive_max_pool1d(
                features,
                output_size=1
            ).squeeze(-1)
            pooled_features.append(pooled)

        combined = torch.cat(pooled_features, dim=1)
        return self.classifier(combined)


class BiGRUClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=192, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_ID
        )
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 192),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(192, 2)
        )

    def forward(self, input_ids, lengths):
        embeddings = self.embedding(input_ids)
        outputs, _ = self.gru(embeddings)

        lengths = lengths.clamp(min=1)
        last_indices = (lengths - 1).view(-1, 1, 1)
        last_indices = last_indices.expand(
            -1,
            1,
            outputs.shape[-1]
        )

        final_hidden = outputs.gather(
            dim=1,
            index=last_indices
        ).squeeze(1)

        return self.classifier(final_hidden)


model_factories = {
    "baseline_mean_pool": lambda: MeanPoolingClassifier(VOCAB_SIZE),
    "experiment_cnn": lambda: CNNClassifier(VOCAB_SIZE),
    "experiment_bigru": lambda: BiGRUClassifier(VOCAB_SIZE),
}

for model_name, factory in model_factories.items():
    test_model = factory().to(DEVICE)
    parameter_count = sum(
        parameter.numel()
        for parameter in test_model.parameters()
    )
    print(model_name, "parameters:", f"{parameter_count:,}")

baseline_mean_pool parameters: 3,856,770
experiment_cnn parameters: 5,181,890
experiment_bigru parameters: 6,057,026


## Architecture and Embedding Justification

### Baseline: Mean-Pooling Classifier

The baseline uses randomly initialized trainable word embeddings followed by masked mean pooling. Padding tokens are excluded from the average. A small feed-forward classifier predicts the sentiment label. This model is intentionally simple and provides a reference point. It captures the overall review meaning but loses word order and detailed phrase structure. The 128-dimensional embedding keeps the baseline compact.

### Experiment A: Multi-Width CNN Classifier

The CNN uses the same from-scratch learned embeddings and applies one-dimensional convolutions with kernel widths 3, 5, and 7. These filters capture short and medium-length sentiment phrases. This changes the architecture relative to the baseline by learning local n-gram patterns. The 160-dimensional embedding gives the convolution layers a richer input representation.

### Experiment B: Bidirectional GRU Classifier

The BiGRU uses trainable embeddings followed by a bidirectional GRU. The forward and backward directions use context from both sides of a token, allowing the model to represent word order and longer-range dependencies. The 192-dimensional embedding and 128 hidden units per direction give this model greater sequence-modeling capacity.

### Embedding Choice

All embeddings are randomly initialized and learned only from the Yelp training split. No pretrained embeddings or pretrained language models are used. Stemming and lemmatization are not applied because the neural embedding layer can learn useful word-form distinctions, and aggressive stemming could remove sentiment-bearing forms.


## Training and Evaluation Functions

Every model is trained independently using cross-entropy loss and AdamW. The checkpoint and history for each model are saved separately.

In [9]:
def synchronize_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    total_examples = 0
    correct = 0
    start_time = time.perf_counter()

    for input_ids, labels, lengths, _ in loader:
        input_ids = input_ids.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        lengths = lengths.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(input_ids, lengths)
        loss = F.cross_entropy(logits, labels)
        loss.backward()

        gradient_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            CONFIG["gradient_clip"]
        )

        optimizer.step()

        batch_size = labels.shape[0]
        total_loss += loss.item() * batch_size
        total_examples += batch_size
        correct += (logits.argmax(dim=1) == labels).sum().item()

    synchronize_device()
    elapsed = time.perf_counter() - start_time

    return {
        "loss": total_loss / total_examples,
        "accuracy": correct / total_examples,
        "examples_per_second": total_examples / max(elapsed, 1e-9),
        "elapsed_seconds": elapsed,
        "gradient_norm": float(gradient_norm.item())
    }


@torch.no_grad()
def predict_model(model, loader):
    model.eval()
    all_probabilities = []
    all_predictions = []
    all_labels = []
    all_tokens = []
    all_lengths = []

    start_time = time.perf_counter()

    for input_ids, labels, lengths, tokens in loader:
        input_ids = input_ids.to(DEVICE, non_blocking=True)
        lengths_device = lengths.to(DEVICE, non_blocking=True)

        logits = model(input_ids, lengths_device)
        probabilities = torch.softmax(logits, dim=1)[:, 1]
        predictions = (probabilities >= 0.5).long()

        all_probabilities.extend(probabilities.cpu().tolist())
        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(labels.tolist())
        all_tokens.extend(list(tokens))
        all_lengths.extend(lengths.tolist())

    synchronize_device()

    return {
        "probabilities": all_probabilities,
        "predictions": all_predictions,
        "labels": all_labels,
        "tokens": all_tokens,
        "lengths": all_lengths,
        "elapsed_seconds": time.perf_counter() - start_time
    }


def train_model(model_name, model):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"]
    )

    history = []
    best_validation_loss = float("inf")
    model_directory = Path("task2_checkpoints")
    model_directory.mkdir(exist_ok=True)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(DEVICE)

    total_start = time.perf_counter()

    for epoch in range(CONFIG["epochs"]):
        train_metrics = train_one_epoch(
            model,
            train_loader,
            optimizer
        )

        validation_predictions = predict_model(
            model,
            validation_loader
        )

        validation_loss = 0.0
        validation_examples = 0

        model.eval()
        with torch.no_grad():
            for input_ids, labels, lengths, _ in validation_loader:
                input_ids = input_ids.to(DEVICE)
                labels = labels.to(DEVICE)
                lengths = lengths.to(DEVICE)
                logits = model(input_ids, lengths)
                loss = F.cross_entropy(logits, labels)
                validation_loss += loss.item() * labels.shape[0]
                validation_examples += labels.shape[0]

        validation_loss /= validation_examples
        validation_accuracy = accuracy_score(
            validation_predictions["labels"],
            validation_predictions["predictions"]
        )

        record = {
            "epoch": epoch + 1,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "validation_loss": validation_loss,
            "validation_accuracy": validation_accuracy,
            "examples_per_second": train_metrics["examples_per_second"],
            "gradient_norm": train_metrics["gradient_norm"],
            "elapsed_seconds": train_metrics["elapsed_seconds"]
        }
        history.append(record)

        checkpoint = {
            "model_name": model_name,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "history": history,
            "config": CONFIG,
            "vocab_size": VOCAB_SIZE,
            "word_to_idx": word_to_idx
        }

        torch.save(
            checkpoint,
            model_directory / f"{model_name}_latest.pt"
        )

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            torch.save(
                checkpoint,
                model_directory / f"{model_name}_best.pt"
            )

        print(
            f"{model_name} | Epoch {epoch + 1}/{CONFIG['epochs']} | "
            f"Train loss: {record['train_loss']:.4f} | "
            f"Val loss: {record['validation_loss']:.4f} | "
            f"Val accuracy: {record['validation_accuracy']:.4f}"
        )

    synchronize_device()
    total_elapsed = time.perf_counter() - total_start
    peak_memory = 0.0

    if torch.cuda.is_available():
        peak_memory = torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 3)

    return {
        "model": model,
        "history": history,
        "total_training_time_seconds": total_elapsed,
        "peak_memory_gb": peak_memory,
        "parameter_count": sum(
            parameter.numel()
            for parameter in model.parameters()
        )
    }

## Train All Three Models

This is the main training cell. It trains the baseline and both experimental models independently. Do not use pretrained weights.

In [10]:
all_model_results = {}

for model_name, factory in model_factories.items():
    print(f"\nStarting {model_name}")

    random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    all_model_results[model_name] = train_model(
        model_name,
        factory()
    )

print("\nAll three models finished training.")


Starting baseline_mean_pool
baseline_mean_pool | Epoch 1/5 | Train loss: 0.3112 | Val loss: 0.2184 | Val accuracy: 0.9148
baseline_mean_pool | Epoch 2/5 | Train loss: 0.2006 | Val loss: 0.1907 | Val accuracy: 0.9271
baseline_mean_pool | Epoch 3/5 | Train loss: 0.1776 | Val loss: 0.1807 | Val accuracy: 0.9313
baseline_mean_pool | Epoch 4/5 | Train loss: 0.1657 | Val loss: 0.1757 | Val accuracy: 0.9328
baseline_mean_pool | Epoch 5/5 | Train loss: 0.1577 | Val loss: 0.1731 | Val accuracy: 0.9344

Starting experiment_cnn
experiment_cnn | Epoch 1/5 | Train loss: 0.2517 | Val loss: 0.1841 | Val accuracy: 0.9259
experiment_cnn | Epoch 2/5 | Train loss: 0.1387 | Val loss: 0.1831 | Val accuracy: 0.9274
experiment_cnn | Epoch 3/5 | Train loss: 0.0872 | Val loss: 0.2019 | Val accuracy: 0.9292
experiment_cnn | Epoch 4/5 | Train loss: 0.0533 | Val loss: 0.2445 | Val accuracy: 0.9243
experiment_cnn | Epoch 5/5 | Train loss: 0.0352 | Val loss: 0.3046 | Val accuracy: 0.9219

Starting experiment_bigru

In [11]:
from pathlib import Path

for model_name, result in all_model_results.items():
    checkpoint_path = Path("task2_checkpoints") / f"{model_name}_best.pt"

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False
    )

    best_model = model_factories[model_name]().to(DEVICE)
    best_model.load_state_dict(checkpoint["model_state_dict"])
    best_model.eval()

    result["model"] = best_model

    print(f"{model_name}: loaded best checkpoint")

baseline_mean_pool: loaded best checkpoint
experiment_cnn: loaded best checkpoint
experiment_bigru: loaded best checkpoint


In [ ]:
# Record the exact hardware used for every model

hardware_summary = {
    "device": str(DEVICE),
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "Not available"
    ),
    "cuda_version": (
        torch.version.cuda
        if torch.cuda.is_available()
        else None
    ),
    "cpu": platform.processor(),
    "operating_system": platform.platform(),
}

for model_name, result in all_model_results.items():
    result["hardware"] = hardware_summary.copy()
    print(model_name)
    print("Device:", hardware_summary["device"])
    print("GPU/CPU:", hardware_summary["gpu"])
    print("CUDA version:", hardware_summary["cuda_version"])
    print()


baseline_mean_pool
Device: cuda
GPU/CPU: NVIDIA GeForce RTX 4090
CUDA version: 13.2

experiment_cnn
Device: cuda
GPU/CPU: NVIDIA GeForce RTX 4090
CUDA version: 13.2

experiment_bigru
Device: cuda
GPU/CPU: NVIDIA GeForce RTX 4090
CUDA version: 13.2



## Required Evaluation Metrics

The next cells calculate accuracy, precision, recall, F1 scores, confusion matrices, ROC-AUC, PR-AUC, MCC, Brier score, calibration error, bootstrap confidence intervals, McNemar tests, and slice-specific performance.

In [13]:
def expected_calibration_error(labels, probabilities, bins=10):
    labels = np.asarray(labels)
    probabilities = np.asarray(probabilities)
    predictions = (probabilities >= 0.5).astype(int)
    ece = 0.0

    for lower in np.linspace(0.0, 1.0, bins + 1)[:-1]:
        upper = lower + 1.0 / bins
        if upper == 1.0:
            mask = (probabilities >= lower) & (probabilities <= upper)
        else:
            mask = (probabilities >= lower) & (probabilities < upper)

        if mask.sum() == 0:
            continue

        confidence = probabilities[mask].mean()
        accuracy = (predictions[mask] == labels[mask]).mean()
        ece += mask.mean() * abs(accuracy - confidence)

    return float(ece)


def bootstrap_confidence_interval(labels, predictions, metric_function, samples=1000):
    rng = np.random.default_rng(SEED)
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    values = []

    for _ in range(samples):
        indices = rng.integers(0, len(labels), len(labels))
        values.append(
            metric_function(labels[indices], predictions[indices])
        )

    return {
        "lower_2_5_percent": float(np.percentile(values, 2.5)),
        "upper_97_5_percent": float(np.percentile(values, 97.5))
    }


def classification_metrics(prediction_result):
    labels = np.asarray(prediction_result["labels"])
    predictions = np.asarray(prediction_result["predictions"])
    probabilities = np.asarray(prediction_result["probabilities"])

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
        labels, predictions, average="micro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0
    )

    metrics = {
        "accuracy": float(accuracy_score(labels, predictions)),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "f1_macro": float(f1_macro),
        "precision_micro": float(precision_micro),
        "recall_micro": float(recall_micro),
        "f1_micro": float(f1_micro),
        "precision_weighted": float(precision_weighted),
        "recall_weighted": float(recall_weighted),
        "f1_weighted": float(f1_weighted),
        "confusion_matrix": confusion_matrix(labels, predictions).tolist(),
        "roc_auc": float(roc_auc_score(labels, probabilities)),
        "pr_auc": float(average_precision_score(labels, probabilities)),
        "mcc": float(matthews_corrcoef(labels, predictions)),
        "brier_score": float(brier_score_loss(labels, probabilities)),
        "expected_calibration_error": expected_calibration_error(labels, probabilities),
        "parameter_count": None,
        "test_examples_per_second": len(labels) / max(prediction_result["elapsed_seconds"], 1e-9)
    }

    metrics["accuracy_ci_95"] = bootstrap_confidence_interval(
        labels, predictions, lambda y, p: accuracy_score(y, p), CONFIG["bootstrap_samples"]
    )
    metrics["macro_f1_ci_95"] = bootstrap_confidence_interval(
        labels, predictions, lambda y, p: f1_score(y, p, average="macro", zero_division=0), CONFIG["bootstrap_samples"]
    )
    metrics["mcc_ci_95"] = bootstrap_confidence_interval(
        labels, predictions, lambda y, p: matthews_corrcoef(y, p), CONFIG["bootstrap_samples"]
    )

    return metrics


evaluation_results = {}

for model_name, result in all_model_results.items():
    predictions = predict_model(
        result["model"],
        test_loader
    )
    metrics = classification_metrics(predictions)
    metrics["parameter_count"] = result["parameter_count"]
    metrics["training_time_seconds"] = result["total_training_time_seconds"]
    metrics["peak_memory_gb"] = result["peak_memory_gb"]
    metrics["hardware"] = result.get("hardware", hardware_summary)
    evaluation_results[model_name] = {
        "predictions": predictions,
        "metrics": metrics
    }

    print(f"\n{model_name}")
    for key, value in metrics.items():
        print(f"{key}: {value}")


baseline_mean_pool
accuracy: 0.9348157894736842
precision_macro: 0.9348186814394852
recall_macro: 0.9348157894736842
f1_macro: 0.9348156810889545
precision_micro: 0.9348157894736842
recall_micro: 0.9348157894736842
f1_micro: 0.9348157894736842
precision_weighted: 0.9348186814394851
recall_weighted: 0.9348157894736842
f1_weighted: 0.9348156810889545
confusion_matrix: [[17737, 1263], [1214, 17786]]
roc_auc: 0.9822201869806094
pr_auc: 0.9824330262022711
mcc: 0.8696344709083607
brier_score: 0.04820229370570147
expected_calibration_error: 0.4364131167919918
parameter_count: 3856770
test_examples_per_second: 16086.120687654722
accuracy_ci_95: {'lower_2_5_percent': 0.9324467105263158, 'upper_97_5_percent': 0.9372631578947368}
macro_f1_ci_95: {'lower_2_5_percent': 0.932439921952528, 'upper_97_5_percent': 0.937259982014954}
mcc_ci_95: {'lower_2_5_percent': 0.864884682647695, 'upper_97_5_percent': 0.8745224178979834}
training_time_seconds: 255.8260255999994
peak_memory_gb: 0.15985107421875
hard

In [14]:
def slice_metrics(prediction_result):
    labels = prediction_result["labels"]
    predictions = prediction_result["predictions"]
    token_lengths = prediction_result["lengths"]
    token_lists = prediction_result["tokens"]

    slices = {
        "short_reviews": [length < 40 for length in token_lengths],
        "medium_reviews": [40 <= length < 100 for length in token_lengths],
        "long_reviews": [length >= 100 for length in token_lengths],
        "contains_negation": [
            any(token in NEGATION_WORDS for token in tokens)
            for tokens in token_lists
        ]
    }

    results = {}
    for slice_name, mask in slices.items():
        selected_labels = [label for label, keep in zip(labels, mask) if keep]
        selected_predictions = [pred for pred, keep in zip(predictions, mask) if keep]

        if not selected_labels:
            continue

        results[slice_name] = {
            "count": len(selected_labels),
            "macro_f1": f1_score(
                selected_labels,
                selected_predictions,
                average="macro",
                zero_division=0
            ),
            "error_rate": 1.0 - accuracy_score(
                selected_labels,
                selected_predictions
            )
        }

    return results


for model_name, result in evaluation_results.items():
    result["slice_metrics"] = slice_metrics(
        result["predictions"]
    )

    print(f"\nSlices for {model_name}")
    for slice_name, values in result["slice_metrics"].items():
        print(slice_name, values)


Slices for baseline_mean_pool
short_reviews {'count': 12899, 'macro_f1': 0.9331418324908021, 'error_rate': 0.06481122567640907}
medium_reviews {'count': 14843, 'macro_f1': 0.9367838430164313, 'error_rate': 0.06319477194637202}
long_reviews {'count': 10258, 'macro_f1': 0.9289396157887138, 'error_rate': 0.06853187755897838}
contains_negation {'count': 23262, 'macro_f1': 0.927841191594811, 'error_rate': 0.0674060699853839}

Slices for experiment_cnn
short_reviews {'count': 12899, 'macro_f1': 0.9257907905876877, 'error_rate': 0.07233118846422204}
medium_reviews {'count': 14843, 'macro_f1': 0.9310236975292885, 'error_rate': 0.0688540052550024}
long_reviews {'count': 10258, 'macro_f1': 0.9218631694104438, 'error_rate': 0.07428348605966073}
contains_negation {'count': 23262, 'macro_f1': 0.9235130924140972, 'error_rate': 0.0704152695383028}

Slices for experiment_bigru
short_reviews {'count': 12899, 'macro_f1': 0.9434194238608475, 'error_rate': 0.05488797581207849}
medium_reviews {'count': 14

## Paired McNemar Tests

McNemar's test compares the baseline's errors with each experimental model's errors on the same test examples.

In [15]:
def mcnemar_test(baseline_result, experimental_result):
    baseline_labels = np.asarray(baseline_result["labels"])
    baseline_predictions = np.asarray(baseline_result["predictions"])
    experimental_predictions = np.asarray(experimental_result["predictions"])

    baseline_correct = baseline_predictions == baseline_labels
    experimental_correct = experimental_predictions == baseline_labels

    b = int(np.sum(baseline_correct & ~experimental_correct))
    c = int(np.sum(~baseline_correct & experimental_correct))

    if b + c == 0:
        statistic = 0.0
        p_value = 1.0
    else:
        statistic = (abs(b - c) - 1) ** 2 / (b + c)
        p_value = math.erfc(math.sqrt(statistic / 2.0))

    return {
        "baseline_only_correct_b": b,
        "experimental_only_correct_c": c,
        "chi_square_with_continuity_correction": statistic,
        "p_value": p_value
    }


baseline_result = evaluation_results["baseline_mean_pool"]["predictions"]

mcnemar_results = {}
for experimental_name in ["experiment_cnn", "experiment_bigru"]:
    mcnemar_results[experimental_name] = mcnemar_test(
        baseline_result,
        evaluation_results[experimental_name]["predictions"]
    )
    print(experimental_name, mcnemar_results[experimental_name])

experiment_cnn {'baseline_only_correct_b': 1376, 'experimental_only_correct_c': 1136, 'chi_square_with_continuity_correction': 22.739251592356688, 'p_value': 1.8553900208668925e-06}
experiment_bigru {'baseline_only_correct_b': 725, 'experimental_only_correct_c': 1184, 'chi_square_with_continuity_correction': 109.88161341016239, 'p_value': 1.0402125956168595e-25}


## Manual Review of 20 Errors

The following cell selects:

- 5 confident false positives
- 5 confident false negatives
- 5 near-threshold errors
- 5 slice-specific errors

The selected errors are exported for manual review. The assigned error type and proposed fix must be checked and edited by the student before submission.

In [18]:
best_model_name = max(
    evaluation_results,
    key=lambda model_name: evaluation_results[model_name]["metrics"]["f1_macro"]
)

print("Best model:", best_model_name)

Best model: experiment_bigru


In [19]:
# Select exactly 20 distinct errors for manual review

review_result = evaluation_results[best_model_name]["predictions"]
labels = np.asarray(review_result["labels"])
predictions = np.asarray(review_result["predictions"])
probabilities = np.asarray(review_result["probabilities"])
tokens_for_review = review_result["tokens"]
lengths_for_review = review_result["lengths"]

selected_indices = set()
error_rows = []


def add_error_rows(indices, category, required_count=5):
    added = 0
    for index in indices:
        index = int(index)
        if index in selected_indices:
            continue

        selected_indices.add(index)
        token_length = int(lengths_for_review[index])
        if token_length < 40:
            slice_name = "short_reviews"
        elif token_length < 100:
            slice_name = "medium_reviews"
        else:
            slice_name = "long_reviews"

        if any(token in NEGATION_WORDS for token in tokens_for_review[index]):
            slice_name += "+contains_negation"

        if category == "confident_false_positive":
            suggested_fix = "Add negation-aware features and inspect positive words used in negative contexts."
        elif category == "confident_false_negative":
            suggested_fix = "Improve vocabulary coverage and test phrase-level features for strong positive expressions."
        elif category == "near_threshold_error":
            suggested_fix = "Improve calibration or use stronger context aggregation."
        else:
            suggested_fix = "Train and evaluate a targeted strategy for the affected data slice."

        error_rows.append({
            "index": index,
            "category": category,
            "error_type": "",
            "actual_label": int(labels[index]),
            "predicted_label": int(predictions[index]),
            "probability_positive": float(probabilities[index]),
            "slice": slice_name,
            "token_length": token_length,
            "review": " ".join(tokens_for_review[index]),
            "observation": "",
            "testable_fix": "",
            "suggested_fix": suggested_fix,
        })
        added += 1
        if added == required_count:
            break

    if added < required_count:
        raise RuntimeError(
            f"Could only select {added} distinct {category} errors; "
            f"the assignment requires {required_count}."
        )


confident_fp = sorted(
    [
        index for index in range(len(labels))
        if labels[index] == 0
        and predictions[index] == 1
        and probabilities[index] >= 0.90
    ],
    key=lambda index: probabilities[index],
    reverse=True,
)
add_error_rows(confident_fp, "confident_false_positive")

confident_fn = sorted(
    [
        index for index in range(len(labels))
        if labels[index] == 1
        and predictions[index] == 0
        and probabilities[index] <= 0.10
    ],
    key=lambda index: probabilities[index],
)
add_error_rows(confident_fn, "confident_false_negative")

near_threshold = sorted(
    [
        index for index in range(len(labels))
        if labels[index] != predictions[index]
        and abs(probabilities[index] - 0.5) <= 0.10
    ],
    key=lambda index: abs(probabilities[index] - 0.5),
)
add_error_rows(near_threshold, "near_threshold_error")

slice_specific = sorted(
    [
        index for index in range(len(labels))
        if labels[index] != predictions[index]
        and (
            len(tokens_for_review[index]) >= 100
            or any(token in NEGATION_WORDS for token in tokens_for_review[index])
        )
    ],
    key=lambda index: lengths_for_review[index],
    reverse=True,
)
add_error_rows(slice_specific, "slice_specific_failure")

if len(error_rows) != 20:
    raise RuntimeError(f"Expected 20 error rows, found {len(error_rows)}.")

with open("task2_error_review.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=error_rows[0].keys())
    writer.writeheader()
    writer.writerows(error_rows)

print("Reviewed model:", best_model_name)
print("Selected error rows:", len(error_rows))
print("Saved task2_error_review.csv")
print("Fill in error_type, observation, and testable_fix manually before submission.")


Reviewed model: experiment_bigru
Selected error rows: 20
Saved task2_error_review.csv
Fill in error_type, observation, and testable_fix manually before submission.


## Manual Error Review Requirement

Open `task2_error_review.csv` and complete the blank columns for all 20 rows:

- `error_type`: classify the failure, such as negation, sarcasm, mixed sentiment, ambiguity, vocabulary limitation, or long-context failure.
- `observation`: explain why the selected model likely made the mistake.
- `testable_fix`: propose one concrete change that can be tested in a future experiment.

The automatically generated `suggested_fix` is only a starting point and does not replace manual analysis.


In [20]:
# Compare the three models using the required test metrics

comparison_summary = []

for model_name, result in evaluation_results.items():
    metrics = result["metrics"]
    comparison_summary.append({
        "model": model_name,
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["f1_macro"],
        "roc_auc": metrics["roc_auc"],
        "pr_auc": metrics["pr_auc"],
        "mcc": metrics["mcc"],
        "brier_score": metrics["brier_score"],
        "ece": metrics["expected_calibration_error"],
        "parameters": metrics["parameter_count"],
        "training_time_seconds": metrics["training_time_seconds"],
        "examples_per_second": metrics["test_examples_per_second"],
        "peak_memory_gb": metrics["peak_memory_gb"],
    })

comparison_summary = sorted(
    comparison_summary,
    key=lambda row: row["macro_f1"],
    reverse=True,
)
best_model_name = comparison_summary[0]["model"]

for row in comparison_summary:
    print(row)

print("Best model by test Macro-F1:", best_model_name)


{'model': 'experiment_bigru', 'accuracy': 0.9468947368421052, 'macro_f1': 0.9468946044463822, 'roc_auc': 0.9869025595567867, 'pr_auc': 0.9870654203830501, 'mcc': 0.8937939302855553, 'brier_score': 0.04117155155324907, 'ece': 0.4629022148170675, 'parameters': 6057026, 'training_time_seconds': 290.1351074000013, 'examples_per_second': 12555.59111871799, 'peak_memory_gb': 1.8029131889343262}
{'model': 'baseline_mean_pool', 'accuracy': 0.9348157894736842, 'macro_f1': 0.9348156810889545, 'roc_auc': 0.9822201869806094, 'pr_auc': 0.9824330262022711, 'mcc': 0.8696344709083607, 'brier_score': 0.04820229370570147, 'ece': 0.4364131167919918, 'parameters': 3856770, 'training_time_seconds': 255.8260255999994, 'examples_per_second': 16086.120687654722, 'peak_memory_gb': 0.15985107421875}
{'model': 'experiment_cnn', 'accuracy': 0.9285, 'macro_f1': 0.9284641230237936, 'roc_auc': 0.9818269847645428, 'pr_auc': 0.9822599225418892, 'mcc': 0.8578609080598568, 'brier_score': 0.053344470512197546, 'ece': 0.4

## Comparative Analysis Notes

The final written comparison must explain which model performed best using the actual test metrics above. Discuss the baseline's simplicity, the CNN's local phrase modeling, and the BiGRU's sequence-context modeling. Report strengths, weaknesses, limitations, and future improvements using observed results rather than generic claims.

The team-level comparison is intentionally left for the final team report because it requires the other member's completed model metrics. Do not invent those results.


## Save the Task 2 Reports

The final cell writes the metrics report, model comparison, McNemar results, configuration, and individual results summary.

In [22]:
output_directory = Path("task2_outputs")
output_directory.mkdir(exist_ok=True)

with open("task2_metrics_report.json", "w", encoding="utf-8") as file:
    json.dump(
        {
            model_name: result["metrics"]
            for model_name, result in evaluation_results.items()
        },
        file,
        indent=2,
    )

with open("task2_slice_metrics.json", "w", encoding="utf-8") as file:
    json.dump(
        {
            model_name: result["slice_metrics"]
            for model_name, result in evaluation_results.items()
        },
        file,
        indent=2,
    )

with open("task2_mcnemar_results.json", "w", encoding="utf-8") as file:
    json.dump(mcnemar_results, file, indent=2)

with open("task2_model_configuration.json", "w", encoding="utf-8") as file:
    json.dump(
        {
            "assignment_identifiers": {
                "SID4": SID4,
                "SEED": SEED,
                "SLICE": SLICE,
                "HP_ID": HP_ID,
                "CLS_A": CLS_A,
                "CLS_B": CLS_B
            },
            "config": CONFIG,
            "models": list(model_factories.keys()),
            "vocabulary_size": VOCAB_SIZE,
            "device": str(DEVICE),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available",
            "hardware_by_model": {
                model_name: result.get("hardware", hardware_summary)
                for model_name, result in all_model_results.items()
            },
            "data_quality": {
                "train": train_quality,
                "test": test_quality,
            }
        },
        file,
        indent=2
    )

comparison_lines = [
    "# DATA266 Task 2: Yelp Polarity Sentiment Classification",
    "",
    "## Individual Work",
    "",
    "Three models were trained independently using learned embeddings initialized from scratch.",
    "No pretrained embeddings or pretrained language models were used.",
    "",
    "## Model Comparison",
    "",
    "| Model | Accuracy | Macro-F1 | ROC-AUC | PR-AUC | MCC | Brier | ECE | Parameters | Training time | Peak memory |",
    "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|",
]

for model_name, result in evaluation_results.items():
    metrics = result["metrics"]
    comparison_lines.append(
        f"| {model_name} | {metrics['accuracy']:.4f} | {metrics['f1_macro']:.4f} | "
        f"{metrics['roc_auc']:.4f} | {metrics['pr_auc']:.4f} | {metrics['mcc']:.4f} | "
        f"{metrics['brier_score']:.4f} | {metrics['expected_calibration_error']:.4f} | "
        f"{metrics['parameter_count']:,} | {metrics['training_time_seconds']:.2f}s | "
        f"{metrics['peak_memory_gb']:.2f} GB |"
    )

comparison_lines.extend([
    "",
    "## Required Analysis Notes",
    "",
    "- The baseline uses masked mean pooling over learned word embeddings.",
    "- Experiment A uses multiple convolution widths to capture local phrases.",
    "- Experiment B uses a bidirectional GRU to model word order and longer context.",
    "- The 20 manually reviewed errors are stored in `task2_error_review.csv`.",
    "- Slice metrics are stored in `task2_slice_metrics.json`.",
    "- McNemar tests are stored in `task2_mcnemar_results.json`.",
    f"- Best model by test Macro-F1: {best_model_name}.",
    f"- Best model test accuracy: {evaluation_results[best_model_name]["metrics"]["accuracy"]:.4f}.",
    "- The CNN and BiGRU should be interpreted using their best validation checkpoints.",
    "- The 20 error rows require manual completion before submission.",
    "- Team-level comparison must be added after the teammate supplies her metrics.",
])

with open("task2_results.md", "w", encoding="utf-8") as file:
    file.write("\n".join(comparison_lines))

print("Saved Task 2 reports:")
print("- task2_metrics_report.json")
print("- task2_slice_metrics.json")
print("- task2_mcnemar_results.json")
print("- task2_model_configuration.json")
print("- task2_results.md")
print("- task2_error_review.csv")

Saved Task 2 reports:
- task2_metrics_report.json
- task2_slice_metrics.json
- task2_mcnemar_results.json
- task2_model_configuration.json
- task2_results.md
- task2_error_review.csv
